In [0]:
from pyspark.sql import functions as F


# These are the raw Bronze datasets produced by the project.
BRONZE_TABLES = {
    "boothwise_votes": (
        "workspace.bronze."
        "trichy_east_boothwise_votes_raw"
    ),
    "candidate_totals": (
        "workspace.bronze."
        "trichy_east_candidate_totals_raw"
    ),
    "polling_booths": (
        "workspace.bronze."
        "trichy_east_polling_booths_raw"
    ),
    "candidate_list": (
        "workspace.bronze."
        "trichy_east_candidate_list_raw"
    )
}


# Confirm every required Bronze table exists.
for dataset_name, table_name in BRONZE_TABLES.items():
    table_exists = spark.catalog.tableExists(table_name)

    print(
        dataset_name,
        "AVAILABLE" if table_exists else "MISSING",
        table_name
    )

    assert table_exists, (
        f"Required Bronze table is missing: {table_name}"
    )


print("Bronze table availability: PASS")

In [0]:
# Build one validation record for the latest batch in each table.
bronze_summary_rows = []


for dataset_name, table_name in BRONZE_TABLES.items():
    table_df = spark.table(table_name)

    # Identify the most recently ingested batch.
    latest_batch_id = (
        table_df
        .groupBy("batch_id")
        .agg(
            F.max("ingestion_timestamp").alias(
                "latest_ingestion_timestamp"
            )
        )
        .orderBy(
            F.col(
                "latest_ingestion_timestamp"
            ).desc()
        )
        .first()["batch_id"]
    )

    latest_batch_df = table_df.filter(
        F.col("batch_id") == latest_batch_id
    )

    # Record coverage and extraction outcomes.
    summary = (
        latest_batch_df
        .agg(
            F.count("*").alias("row_count"),
            F.countDistinct("source_file_path").alias(
                "source_file_count"
            ),
            F.countDistinct("election_year").alias(
                "election_year_count"
            ),
            F.sum(
                F.when(
                    F.col("parse_status") == "SUCCESS",
                    1
                ).otherwise(0)
            ).alias("success_rows"),
            F.sum(
                F.when(
                    F.col("parse_status") == "FAILED",
                    1
                ).otherwise(0)
            ).alias("failed_rows")
        )
        .first()
    )

    bronze_summary_rows.append({
        "dataset_name": dataset_name,
        "table_name": table_name,
        "latest_batch_id": latest_batch_id,
        "row_count": summary["row_count"],
        "source_file_count": summary["source_file_count"],
        "election_year_count": summary["election_year_count"],
        "success_rows": summary["success_rows"],
        "failed_rows": summary["failed_rows"]
    })


bronze_latest_batch_summary_df = spark.createDataFrame(
    bronze_summary_rows
)

display(
    bronze_latest_batch_summary_df
    .orderBy("dataset_name")
)

In [0]:
# PASS means complete coverage with no failed records.
# WARN means coverage exists, but Bronze retained extraction failures.
# FAIL means a required year/source is missing.
bronze_validation_df = (
    bronze_latest_batch_summary_df
    .withColumn(
        "validation_status",
        F.when(
            (F.col("source_file_count") != 3)
            | (F.col("election_year_count") != 3)
            | (F.col("success_rows") == 0),
            F.lit("FAIL")
        )
        .when(
            F.col("failed_rows") > 0,
            F.lit("WARN")
        )
        .otherwise(F.lit("PASS"))
    )
)


display(
    bronze_validation_df
    .select(
        "dataset_name",
        "row_count",
        "source_file_count",
        "election_year_count",
        "success_rows",
        "failed_rows",
        "validation_status"
    )
    .orderBy("dataset_name")
)


# Stop the validation notebook only for incomplete coverage.
failed_dataset_count = (
    bronze_validation_df
    .filter(F.col("validation_status") == "FAIL")
    .count()
)

assert failed_dataset_count == 0, (
    "At least one Bronze dataset has incomplete coverage."
)

print("Bronze dataset coverage validation: PASS")

In [0]:
# Produce detailed coverage evidence for every dataset and year.
yearly_coverage_frames = []

for dataset_name, table_name in BRONZE_TABLES.items():
    latest_batch_id = (
        bronze_validation_df
        .filter(F.col("dataset_name") == dataset_name)
        .select("latest_batch_id")
        .first()["latest_batch_id"]
    )

    yearly_summary_df = (
        spark.table(table_name)
        .filter(F.col("batch_id") == latest_batch_id)
        .groupBy(
            "election_year",
            "source_file_name"
        )
        .agg(
            F.count("*").alias("row_count"),
            F.sum(
                F.when(
                    F.col("parse_status") == "SUCCESS",
                    1
                ).otherwise(0)
            ).alias("success_rows"),
            F.sum(
                F.when(
                    F.col("parse_status") == "FAILED",
                    1
                ).otherwise(0)
            ).alias("failed_rows")
        )
        .withColumn(
            "dataset_name",
            F.lit(dataset_name)
        )
        .withColumn(
            "latest_batch_id",
            F.lit(latest_batch_id)
        )
    )

    yearly_coverage_frames.append(yearly_summary_df)


bronze_yearly_coverage_df = yearly_coverage_frames[0]

for coverage_df in yearly_coverage_frames[1:]:
    bronze_yearly_coverage_df = (
        bronze_yearly_coverage_df.unionByName(coverage_df)
    )


display(
    bronze_yearly_coverage_df
    .select(
        "dataset_name",
        "election_year",
        "source_file_name",
        "row_count",
        "success_rows",
        "failed_rows",
        "latest_batch_id"
    )
    .orderBy(
        "dataset_name",
        "election_year"
    )
)

print("Expected dataset/year combinations: 12")
print(
    "Actual dataset/year combinations:",
    bronze_yearly_coverage_df.count()
)

In [0]:
# Confirm all four datasets cover all three election years.
dataset_year_count = bronze_yearly_coverage_df.count()

missing_success_count = (
    bronze_yearly_coverage_df
    .filter(F.col("success_rows") == 0)
    .count()
)

assert dataset_year_count == 12, (
    "Expected 4 datasets across 3 election years."
)

assert missing_success_count == 0, (
    "At least one dataset/year has no successful records."
)


print("Bronze datasets validated: 4")
print("Election years validated: 2016, 2021, 2026")
print("Dataset/year combinations validated:", dataset_year_count)
print("Bronze layer validation: PASS")